<a href="https://colab.research.google.com/github/davo300/Llama-Chatbot/blob/main/finetune_tinyllama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================
#  STEP 0: Environment Setup
# ==============================================================

!pip install -q torch transformers peft accelerate bitsandbytes datasets unsloth

# Optional: check GPU
!nvidia-smi


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.2/272.2 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 145.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 21.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [ ]:
# ==============================================================
#  STEP 1: Imports & Dataset
# ==============================================================

from unsloth import FastLanguageModel
from datasets import load_dataset

# Upload your dataset file (JSONL) to Colab or mount Drive first.
# Example: upload "leetcode_helper.jsonl" in the left sidebar (Files tab).
DATA_PATH = "/content/leetcode_helper.jsonl" # or "compiler_helper.jsonl"
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

print("Dataset loaded:", dataset)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded: Dataset({
    features: ['instruction', 'response'],
    num_rows: 20
})


In [ ]:
# ==============================================================
#  STEP 2: Load Base Model (Gemma 1B)
# ==============================================================

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
model, tokenizer = FastLanguageModel.from_pretrained(model_name)

==((====))==  Unsloth 2025.10.11: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
# ==============================================================
#  STEP 3: Apply LoRA Configuration
# ==============================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=8,                       # LoRA rank (adapter size)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","v_proj"],  # safe attention targets
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [ ]:
# ==============================================================
#  STEP 4: Training Configuration
# ==============================================================

from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import torch

# Tokenize your dataset
def tokenize_function(example):
    full_text = example["instruction"] + "\n" + example["response"]
    tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=False)

# Define training arguments
training_args = TrainingArguments(
    output_dir="lora-tinyllama",
    per_device_train_batch_size=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

# Collator for causal LM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained("lora-tinyllama")
tokenizer.save_pretrained("lora-tinyllama")


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

/tmp/ipython-input-1813340369.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 2 | Total steps = 20
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 1,126,400 of 1,101,174,784 (0.10% trained)


Step,Training Loss
10,2.858900
20,2.697000


Unsloth: Will smartly offload gradients to save VRAM!


('lora-tinyllama/tokenizer_config.json',
 'lora-tinyllama/special_tokens_map.json',
 'lora-tinyllama/chat_template.jinja',
 'lora-tinyllama/tokenizer.model',
 'lora-tinyllama/added_tokens.json',
 'lora-tinyllama/tokenizer.json')

In [ ]:

# ==============================================================
#  STEP 5: Save Outputs
# ==============================================================


# Zip for download
!zip -r lora-tinyllama.zip lora-tinyllama

from google.colab import files
files.download("lora-tinyllama.zip")


  adding: lora-tinyllama/ (stored 0%)
  adding: lora-tinyllama/chat_template.jinja (deflated 60%)
  adding: lora-tinyllama/tokenizer.json (deflated 85%)
  adding: lora-tinyllama/checkpoint-20/ (stored 0%)
  adding: lora-tinyllama/checkpoint-20/chat_template.jinja (deflated 60%)
  adding: lora-tinyllama/checkpoint-20/tokenizer.json (deflated 85%)
  adding: lora-tinyllama/checkpoint-20/tokenizer_config.json (deflated 69%)
  adding: lora-tinyllama/checkpoint-20/trainer_state.json (deflated 58%)
  adding: lora-tinyllama/checkpoint-20/tokenizer.model (deflated 55%)
  adding: lora-tinyllama/checkpoint-20/scaler.pt (deflated 64%)
  adding: lora-tinyllama/checkpoint-20/adapter_config.json (deflated 55%)
  adding: lora-tinyllama/checkpoint-20/README.md (deflated 65%)
  adding: lora-tinyllama/checkpoint-20/special_tokens_map.json (deflated 79%)
  adding: lora-tinyllama/checkpoint-20/adapter_model.safetensors (deflated 7%)
  adding: lora-tinyllama/checkpoint-20/rng_state.pth (deflated 26%)
  addi

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1️⃣ Install the latest Unsloth + Zoo
!pip install -q --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 219.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 208.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 141.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 255.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 274.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.1/75.1 kB 199.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 419.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 435.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 304.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 433.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q "numpy<2.1" "pandas==2.2.2" "protobuf<6.0" "pillow<12.0" --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 129.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 114.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 34.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
bigframes 2.27.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompa

In [ ]:
!pip install -U --force-reinstall --no-cache-dir --no-deps unsloth unsloth_zoo


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.5/61.5 kB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.7/348.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.6/273.6 kB 86.6 MB/s eta 0:00:00
  Attempting uninstall: unsloth_zoo
    Found existing installation: unsloth_zoo 2025.10.13
    Uninstalling unsloth_zoo-2025.10.13:
      Successfully uninstalled unsloth_zoo-2025.10.13
  Attempting uninstall: unsloth
    Found existing installation: unsloth 2025.10.12
    Uninstalling unsloth-2025.10.12:
      Successfully uninstalled unsloth-2025.10.12


In [ ]:
!ls -lh


total 8.3M
-rw-r--r-- 1 root root 1002 Oct 30 18:42 adapter_config.json
-rw-r--r-- 1 root root 4.4M Oct 30 18:42 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Oct 30 18:42 chat_template.jinja
-rw-r--r-- 1 root root 5.2K Oct 30 18:42 README.md
drwxr-xr-x 1 root root 4.0K Oct 29 13:38 sample_data
-rw-r--r-- 1 root root  552 Oct 30 18:42 special_tokens_map.json
-rw-r--r-- 1 root root  952 Oct 30 18:42 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Oct 30 18:42 tokenizer.json
-rw-r--r-- 1 root root 489K Oct 30 18:42 tokenizer.model
drwxr-xr-x 4 root root 4.0K Oct 30 18:43 unsloth_compiled_cache


In [ ]:
!ls -lh /content/lora-tinyllama


ls: cannot access '/content/lora-tinyllama': No such file or directory


In [ ]:
!unzip -q /content/lora-tinyllama.zip -d /content/
!ls -lh /content/lora-tinyllama


total 8.3M
-rw-r--r-- 1 root root 1002 Oct 30 01:54 adapter_config.json
-rw-r--r-- 1 root root 4.4M Oct 30 01:54 adapter_model.safetensors
-rw-r--r-- 1 root root  410 Oct 30 01:54 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Oct 30 01:54 checkpoint-10
drwxr-xr-x 2 root root 4.0K Oct 30 01:54 checkpoint-20
-rw-r--r-- 1 root root 5.2K Oct 30 01:54 README.md
-rw-r--r-- 1 root root  552 Oct 30 01:54 special_tokens_map.json
-rw-r--r-- 1 root root  952 Oct 30 01:54 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Oct 30 01:54 tokenizer.json
-rw-r--r-- 1 root root 489K Oct 30 01:54 tokenizer.model


In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# 1️⃣ Load the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    dtype="float16",
    load_in_4bit=False,
)

# 2️⃣ Load the LoRA adapter on top of it
lora_model = PeftModel.from_pretrained(model, "/content/lora-tinyllama")

# 3️⃣ Merge LoRA weights into the base model
lora_model = lora_model.merge_and_unload()

# 4️⃣ Save the merged model
save_path = "/content/tinyllama-lora-merged"
lora_model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("✅ Successfully merged and saved to:", save_path)



==((====))==  Unsloth 2025.10.12: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Successfully merged and saved to: /content/tinyllama-lora-merged


In [ ]:
!ls -lh /content/tinyllama-lora-merged


total 2.1G
-rw-r--r-- 1 root root  410 Oct 30 19:39 chat_template.jinja
-rw-r--r-- 1 root root  724 Oct 30 19:36 config.json
-rw-r--r-- 1 root root  124 Oct 30 19:36 generation_config.json
-rw-r--r-- 1 root root 2.1G Oct 30 19:39 model.safetensors
-rw-r--r-- 1 root root  552 Oct 30 19:39 special_tokens_map.json
-rw-r--r-- 1 root root  951 Oct 30 19:39 tokenizer_config.json
-rw-r--r-- 1 root root 3.5M Oct 30 19:39 tokenizer.json
-rw-r--r-- 1 root root 489K Oct 30 19:39 tokenizer.model


In [ ]:
!zip -r tinyllama-lora-merged.zip /content/tinyllama-lora-merged


  adding: content/tinyllama-lora-merged/ (stored 0%)
  adding: content/tinyllama-lora-merged/model.safetensors


zip error: Interrupted (aborting)


In [ ]:
from unsloth import FastLanguageModel

prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Valid Anagram\nHow do we determine if two strings are anagrams?"}],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=400)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


<|user|>
Valid Anagram
How do we determine if two strings are anagrams? 
<|assistant|>
To determine if two strings are anagrams, we can use the following algorithm:

1. Compare the first character of each string.
2. If the first character is the same, then the strings are anagrams.
3. If the first character is different, then the strings are not anagrams.

For example, the strings "abcd" and "abcde" are not anagrams because the first character (a) is different.

Here's an example implementation in Python:

```python
def is_anagram(s1, s2):
    # Compare the first character of each string
    for I in range(len(s1)):
        if s1[i] != s2[i]:
            return False
    return True
```

This implementation uses a loop to compare the first characters of the two strings. If the first characters are different, then the strings are not anagrams.


In [2]:
# Clone your repo
!git clone https://github.com/davo300/Llama-Chatbot.git
%cd llm-chatbot-project

Cloning into 'Llama-Chatbot'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 25 (delta 5), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 8.13 KiB | 8.13 MiB/s, done.
Resolving deltas: 100% (5/5), done.
[Errno 2] No such file or directory: 'llm-chatbot-project'
/content


In [6]:
# Copy your notebook there
!cp /content/finetune_tinyllama.ipynb .

cp: cannot stat '/content/finetune_tinyllama.ipynb': No such file or directory


In [5]:
!ls /content


Llama-Chatbot  sample_data
